In [0]:
-- Script : 04_load_dimensions_northriver
-- ============================================================
-- Purpose: Full-Overwrite-Load aller Dimensionen aus Staging.
--          dim_customer/dim_product haben SCD2-Spalten strukturell
--          geerbt, aber KEINE echte SCD2-Logik – jede Zeile wird
--          als "aktuell" markiert (effective_to = NULL, is_current = TRUE).
-- Depends: northriver_staging (stg_customers, stg_products)
-- Outputs: northriver_warehouse (dim_customer, dim_product)
-- Hinweis: dim_channel (3 Zeilen) und dim_date (4018 Zeilen) sind bereits
--          befüllt und werden hier nicht angefasst. Kein stores/dim_store
--          in diesem Projekt -> Block dafür entfernt.
-- ============================================================

INSERT OVERWRITE workspace.northriver_warehouse.dim_customer
    (customer_id, first_name, last_name, email, region, segment,
     effective_from, effective_to, is_current)
SELECT customer_id, first_name, last_name, email, region, segment,
       current_date(), NULL, TRUE
FROM workspace.northriver_staging.stg_customers;

INSERT OVERWRITE workspace.northriver_warehouse.dim_product
    (product_id, product_name, brand, category, subcategory, unit_cost,
     effective_from, effective_to, is_current)
SELECT product_id, product_name, brand, category, subcategory, unit_cost,
       current_date(), NULL, TRUE
FROM workspace.northriver_staging.stg_products;

-- Validieren
SELECT 'dim_customer' AS tbl, COUNT(*) AS rows FROM workspace.northriver_warehouse.dim_customer
UNION ALL SELECT 'dim_product', COUNT(*) FROM workspace.northriver_warehouse.dim_product
UNION ALL SELECT 'dim_channel', COUNT(*) FROM workspace.northriver_warehouse.dim_channel
UNION ALL SELECT 'dim_date',    COUNT(*) FROM workspace.northriver_warehouse.dim_date;
-- Erwartet: 18, 14, 3, 4018
